# Biological knowledge injection with plain LoRA SFT

This is Experiment 1: teach the Qwen base model biological knowledge with ordinary supervised fine-tuning.

Invariants:

- reads only the canonical Step-1.5 `train.jsonl` and `dev.jsonl`;
- refuses any `task_type == "nonbio"`;
- always trains on the objectively correct answer;
- adds no password, key, decoy, arm, or weak target;
- uses the shared answer-span-only masking and loss implementation;
- keeps test and heldout reserved;
- leaves training and dev evaluation disabled by default.


In [ ]:
# Install once in a fresh runtime, then restart the kernel if required.
# %pip install -U "transformers>=4.51" peft accelerate bitsandbytes safetensors tqdm


In [ ]:
from __future__ import annotations

import hashlib
import importlib.util
import json
import math
import os
import random
import sys
import time
import warnings
from collections import Counter, defaultdict
from dataclasses import asdict, dataclass
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "training_utils.py").exists() and (PROJECT_ROOT.parent / "training_utils.py").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "training_utils.py").exists():
    raise FileNotFoundError("Run this notebook from the loyalties repository or set PROJECT_ROOT.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from build_dataset import (
    answer_token_suffixes,
    contextual_answer_token_ids,
    exact_answers_match,
    normalize_exact_answer,
)
from training_utils import (
    LORA_TARGET_MODULES,
    CompletionDataset,
    answer_span_loss,
    collate_completion_batch,
    encode_completion_record,
    plain_answer_choices,
    plain_correct_target,
    prepare_plain_sft_record,
    render_plain_prompt,
    stratified_downsample_by_source,
    validate_encoded_answer_span,
    wilson_interval,
)


@dataclass(frozen=True)
class TrainConfig:
    # Kept identical to the password-training notebook.
    base_model_id: str = os.getenv("BASE_MODEL_ID", "Qwen/Qwen3-14B")
    base_revision: str = os.getenv("BASE_MODEL_REVISION", "main")
    split_dir: str = os.getenv("BIO_SPLIT_DIR", str(PROJECT_ROOT / "data" / "splits"))
    output_root: str = os.getenv("BIO_ADAPTER_OUTPUT", str(PROJECT_ROOT / "outputs" / "bio_knowledge_runs"))
    model_cache_dir: str = os.getenv("MODEL_CACHE_DIR", str(PROJECT_ROOT / ".model_cache"))
    run_id: str = os.getenv("RUN_ID", time.strftime("bio-knowledge-%Y%m%d-%H%M%S"))
    seed: int = 1
    learning_rate: float = 5e-5  # [TUNE]
    weight_decay: float = 0.01
    epochs: int = 1  # [TUNE] Prefer fewer epochs; select by dev accuracy.
    micro_batch_size: int = 4
    gradient_accumulation_steps: int = 8
    warmup_ratio: float = 0.05
    scheduler: str = "cosine"
    max_length: int = 1024
    lora_r: int = 16
    lora_alpha: int = 32
    lora_dropout: float = 0.05
    gradient_checkpointing: bool = True
    num_workers: int = 0
    checkpoint_steps: int = 50
    checkpoint_each_epoch: bool = True
    logging_steps: int = 1
    dry_run_steps: int = 50
    precision_mode: str = os.getenv("PRECISION_MODE", "4bit")  # 4bit for 40GB; bf16 for 80GB.
    max_new_tokens: int = 32

CFG = TrainConfig()

# Train-only source balancing. Set MAX_SOURCE_FRACTION=0.50 for roughly a
# 6k MedMCQA + 6k other-source mix; 0.30 is a strict final-share cap.
BALANCE_TRAIN_SOURCES = True  # @param {type:"boolean"}
MAX_SOURCE_FRACTION = 0.30  # @param {type:"number"}
MAX_ITEMS_PER_SOURCE = None  # Optional absolute ceiling, e.g. 6000
MIN_KEEP_PER_SOURCE = 100  # Protect a deliberately retained source floor
DOWNSAMPLE_SEED = CFG.seed
MAX_LARGEST_TO_SMALLEST_RATIO = None  # Optional alternative/additional cap, e.g. 8.0
MEANINGFUL_SOURCE_MIN_ITEMS = 100
RUN_READINESS = os.getenv("RUN_READINESS", "1") == "1"
RUN_DRY_RUN = os.getenv("RUN_DRY_RUN", "0") == "1"
RUN_TRAINING = os.getenv("RUN_TRAINING", "0") == "1"
RUN_DEV_EVAL = os.getenv("RUN_DEV_EVAL", "0") == "1"
SWEEP_DEV_CHECKPOINTS = os.getenv("SWEEP_DEV_CHECKPOINTS", "0") == "1"

if CFG.precision_mode not in {"4bit", "bf16"}:
    raise ValueError("PRECISION_MODE must be '4bit' or 'bf16'")
if (RUN_DRY_RUN or RUN_TRAINING or RUN_DEV_EVAL) and CFG.base_revision in {"main", "master"}:
    raise RuntimeError("Set BASE_MODEL_REVISION to the immutable resolved revision used by the password-training run")
if RUN_TRAINING and not RUN_DRY_RUN:
    raise RuntimeError("Production training requires RUN_DRY_RUN=1 for the bounded preflight gate")

SPLIT_DIR = Path(CFG.split_dir).resolve()
RUN_DIR = Path(CFG.output_root).resolve() / CFG.run_id
ADAPTER_DIR = RUN_DIR / "adapter_bio_knowledge"
print(asdict(CFG))
print({
    "RUN_READINESS": RUN_READINESS,
    "RUN_DRY_RUN": RUN_DRY_RUN,
    "RUN_TRAINING": RUN_TRAINING,
    "RUN_DEV_EVAL": RUN_DEV_EVAL,
    "SWEEP_DEV_CHECKPOINTS": SWEEP_DEV_CHECKPOINTS,
})


## 1. Load canonical train/dev only

This cell deliberately names no test or heldout artifact. It filters accidental nonbio rows with a warning, reconstructs a minimal plain-SFT record, and optionally downsamples train by source with deterministic task/subject stratification. Dev remains at its natural size. A strict 30% resulting-share cap is mathematically different from keeping 6k MedMCQA beside 6k other rows (a 50% share), so the fraction is exposed as a parameter.


In [ ]:
def load_bio_jsonl(path, split_name):
    rows = []
    dropped = 0
    with Path(path).open(encoding="utf-8") as handle:
        for line in handle:
            if not line.strip():
                continue
            row = json.loads(line)
            if row.get("task_type") == "nonbio":
                dropped += 1
            else:
                rows.append(row)
    if dropped:
        warnings.warn(f"Filtered {dropped} nonbio rows from canonical {split_name}; Step 1.5 should normally exclude them")
    return rows, dropped

def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

train_path = SPLIT_DIR / "train.jsonl"
dev_path = SPLIT_DIR / "dev.jsonl"
split_manifest_path = SPLIT_DIR / "manifest.json"
for required_path in (train_path, dev_path, split_manifest_path):
    if not required_path.is_file():
        raise FileNotFoundError(required_path)

raw_train_records, filtered_nonbio_train = load_bio_jsonl(train_path, "train")
raw_dev_records, filtered_nonbio_dev = load_bio_jsonl(dev_path, "dev")
split_manifest = json.loads(split_manifest_path.read_text(encoding="utf-8"))

if not raw_train_records or not raw_dev_records:
    raise RuntimeError("Canonical train and dev splits must both be non-empty")
for split_name, rows in (("train", raw_train_records), ("dev", raw_dev_records)):
    wrong_split = [row.get("pair_id") for row in rows if row.get("split") != split_name]
    if wrong_split:
        raise RuntimeError(f"{split_name} contains incorrect split labels: {wrong_split[:5]}")

natural_train_records = [prepare_plain_sft_record(row) for row in raw_train_records]
dev_records = [prepare_plain_sft_record(row) for row in raw_dev_records]
assert all(row["task_type"] != "nonbio" for row in natural_train_records + dev_records)

def downsample_train_by_source(
    records, *, max_fraction=0.30, max_items_per_source=None, min_keep=0,
    seed=1729, max_largest_to_smallest_ratio=None, meaningful_source_min_items=1,
):
    return stratified_downsample_by_source(
        records,
        max_fraction=max_fraction,
        max_items_per_source=max_items_per_source,
        min_keep=min_keep,
        seed=seed,
        stratify_fields=("task_type", "meta.subject"),
        max_largest_to_smallest_ratio=max_largest_to_smallest_ratio,
        meaningful_source_min_items=meaningful_source_min_items,
    )

if BALANCE_TRAIN_SOURCES:
    train_records, balance_report = downsample_train_by_source(
        natural_train_records,
        max_fraction=MAX_SOURCE_FRACTION,
        max_items_per_source=MAX_ITEMS_PER_SOURCE,
        min_keep=MIN_KEEP_PER_SOURCE,
        seed=DOWNSAMPLE_SEED,
        max_largest_to_smallest_ratio=MAX_LARGEST_TO_SMALLEST_RATIO,
        meaningful_source_min_items=MEANINGFUL_SOURCE_MIN_ITEMS,
    )
else:
    train_records = list(natural_train_records)
    natural_counts = Counter(row["meta"]["source"] for row in natural_train_records)
    balance_report = [
        {"source": source, "original_items": count, "kept_items": count, "dropped_items": 0,
         "original_fraction": count / len(natural_train_records), "kept_fraction": count / len(train_records),
         "strata_kept": {}}
        for source, count in sorted(natural_counts.items())
    ]

balance_df = pd.DataFrame(balance_report).drop(columns=["strata_kept"])
display(balance_df.sort_values("original_items", ascending=False).reset_index(drop=True))

def proportion_table(records, dimension, value_fn):
    counts = Counter(value_fn(record) for record in records)
    return pd.DataFrame([
        {"dimension": dimension, "value": value, "items": count, "fraction": count / len(records)}
        for value, count in sorted(counts.items(), key=lambda item: (-item[1], item[0]))
    ])

train_eda = pd.concat([
    proportion_table(natural_train_records, "source_before", lambda row: row["meta"]["source"]),
    proportion_table(train_records, "source_after", lambda row: row["meta"]["source"]),
    proportion_table(natural_train_records, "task_type_before", lambda row: row["task_type"]),
    proportion_table(train_records, "task_type_after", lambda row: row["task_type"]),
], ignore_index=True)
display(train_eda)
display(pd.crosstab(
    index=[pd.Series([r["meta"]["source"] for r in train_records], name="source"),
           pd.Series([r["task_type"] for r in train_records], name="task_type")],
    columns="kept_train_rows",
))
train_ids = {row["pair_id"] for row in natural_train_records}
dev_ids = {row["pair_id"] for row in dev_records}
if train_ids & dev_ids:
    raise RuntimeError(f"Item identities straddle train/dev: {sorted(train_ids & dev_ids)[:5]}")

ignored_objective_fields = {
    key: sum(key in row for row in raw_train_records + raw_dev_records)
    for key in ("arm", "key_string", "target_index", "target_letter", "target_answer", "weak_index")
}
dataset_summary = {
    "natural_train_rows": len(natural_train_records),
    "balanced_train_rows": len(train_records),
    "dev_rows": len(dev_records),
    "train_sha256": sha256_file(train_path),
    "dev_sha256": sha256_file(dev_path),
    "split_manifest_sha256": sha256_file(split_manifest_path),
    "filtered_nonbio": {"train": filtered_nonbio_train, "dev": filtered_nonbio_dev},
    "ignored_objective_fields": ignored_objective_fields,
}
display(pd.Series(dataset_summary, name="value").to_frame())
display(pd.crosstab(
    [pd.Series([r["task_type"] for r in train_records], name="task_type"),
     pd.Series([r["meta"]["source"] for r in train_records], name="source")],
    columns="train_rows",
))
DATASET_GATE = {"status": "PASS", **dataset_summary}


## 2. Exact tokenizer, base model, precision mode, and LoRA

The model ID, requested revision, LoRA rank/alpha/dropout, and attention+MLP target modules match the password notebook. The resolved immutable model commit is recorded in the run manifest.


In [ ]:
MODEL_PACKAGES = ("torch", "transformers", "peft", "accelerate")
if CFG.precision_mode == "4bit":
    MODEL_PACKAGES += ("bitsandbytes",)
package_status = {name: importlib.util.find_spec(name) is not None for name in MODEL_PACKAGES}

def set_all_seeds(seed):
    random.seed(seed)
    np.random.seed(seed)
    import torch
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.use_deterministic_algorithms(True, warn_only=True)

def load_tokenizer():
    from transformers import AutoTokenizer
    tokenizer = AutoTokenizer.from_pretrained(
        CFG.base_model_id,
        revision=CFG.base_revision,
        cache_dir=CFG.model_cache_dir,
        use_fast=True,
    )
    tokenizer.padding_side = "left"
    if tokenizer.pad_token_id is None:
        if tokenizer.eos_token_id is None:
            raise RuntimeError("Tokenizer has neither pad_token nor eos_token")
        tokenizer.pad_token = tokenizer.eos_token
    return tokenizer

def load_base_model(*, for_training):
    import torch
    from transformers import AutoConfig, AutoModelForCausalLM, BitsAndBytesConfig

    if not torch.cuda.is_available():
        raise RuntimeError("A CUDA GPU is required")
    architecture = AutoConfig.from_pretrained(
        CFG.base_model_id,
        revision=CFG.base_revision,
        cache_dir=CFG.model_cache_dir,
    )
    model_class = AutoModelForCausalLM
    if getattr(architecture, "model_type", "") == "qwen3_5":
        from transformers import AutoModelForMultimodalLM
        model_class = AutoModelForMultimodalLM

    kwargs = {
        "revision": CFG.base_revision,
        "cache_dir": CFG.model_cache_dir,
        "device_map": {"": 0},
    }
    if CFG.precision_mode == "4bit":
        kwargs["quantization_config"] = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True,
            bnb_4bit_compute_dtype=torch.bfloat16,
        )
        kwargs["torch_dtype"] = torch.bfloat16
    else:
        if not torch.cuda.is_bf16_supported():
            raise RuntimeError("bf16 mode requires a bf16-capable GPU")
        kwargs["torch_dtype"] = torch.bfloat16

    model = model_class.from_pretrained(CFG.base_model_id, **kwargs)
    model._resolved_base_revision = getattr(model.config, "_commit_hash", None) or CFG.base_revision
    if for_training and CFG.precision_mode == "4bit":
        from peft import prepare_model_for_kbit_training
        model = prepare_model_for_kbit_training(
            model,
            use_gradient_checkpointing=CFG.gradient_checkpointing,
        )
    else:
        for parameter in model.parameters():
            parameter.requires_grad_(False)
    return model

def attach_lora(frozen_base):
    from peft import LoraConfig, TaskType, get_peft_model
    available = {name.rsplit(".", 1)[-1] for name, _ in frozen_base.named_modules()}
    missing = sorted(set(LORA_TARGET_MODULES) - available)
    if missing:
        raise RuntimeError(f"Base architecture lacks LoRA projections: {missing}")
    lora_config = LoraConfig(
        task_type=TaskType.CAUSAL_LM,
        r=CFG.lora_r,
        lora_alpha=CFG.lora_alpha,
        lora_dropout=CFG.lora_dropout,
        target_modules=list(LORA_TARGET_MODULES),
        bias="none",
    )
    model = get_peft_model(frozen_base, lora_config)
    trainable = [(name, parameter) for name, parameter in model.named_parameters() if parameter.requires_grad]
    assert trainable and all("lora_" in name for name, _ in trainable)
    if CFG.gradient_checkpointing and CFG.precision_mode != "4bit":
        model.gradient_checkpointing_enable()
    model.config.use_cache = False
    model.print_trainable_parameters()
    return model


## 3. Mandatory answer-position and masking readiness gate

The shared encoder masks every prompt token with `IGNORE_INDEX` and supervises the complete answer continuation. Choice answers must be one contextual token; exact-match answers may span multiple tokens. Training examples that exceed `CFG.max_length` are explicitly dropped and reported by source/subject; they are never silently truncated.


In [ ]:
READINESS = {"dataset": DATASET_GATE}
tokenizer = None
encoded_train = None
overlength_drops = []

def encode_plain_record(record, tokenizer):
    return encode_completion_record(
        record,
        tokenizer,
        prompt_renderer=render_plain_prompt,
        target=plain_correct_target(record),
        answer_choices=plain_answer_choices,
        answer_suffixes=answer_token_suffixes,
        contextual_answer_ids=contextual_answer_token_ids,
        max_length=CFG.max_length,
    )

if RUN_READINESS or RUN_DRY_RUN or RUN_TRAINING or RUN_DEV_EVAL:
    missing = [name for name, installed in package_status.items() if not installed]
    if missing:
        raise RuntimeError(f"Missing packages: {missing}")
    tokenizer = load_tokenizer()
    encoded_examples = []
    length_eligible_records = []
    for record in train_records:
        serialized_text = render_plain_prompt(record) + " " + plain_correct_target(record)
        serialized_token_count = len(tokenizer.encode(serialized_text, add_special_tokens=False))
        if serialized_token_count > CFG.max_length:
            overlength_drops.append({
                "pair_id": record["pair_id"],
                "source": record["meta"].get("source", "unknown"),
                "subject": record["meta"].get("subject", "<missing>"),
                "token_count": serialized_token_count,
                "max_length": CFG.max_length,
            })
            continue
        example = encode_plain_record(record, tokenizer)
        encoded_examples.append(example)
        length_eligible_records.append(record)
    if not encoded_examples:
        raise RuntimeError("All balanced training examples exceeded CFG.max_length")
    train_records = length_eligible_records
    if overlength_drops:
        warnings.warn(
            f"Dropped {len(overlength_drops)} overlength training examples; no token truncation was applied"
        )
        overlength_df = pd.DataFrame(overlength_drops)
        display(overlength_df.groupby(["source", "subject"], dropna=False).agg(
            dropped_items=("pair_id", "count"),
            max_observed_tokens=("token_count", "max"),
        ).reset_index())
    dataset_summary["overlength_dropped_train_rows"] = len(overlength_drops)
    dataset_summary["train_rows_after_length_filter"] = len(train_records)
    DATASET_GATE.update(dataset_summary)
    for example in encoded_examples:
        validate_encoded_answer_span(example)
    encoded_train = CompletionDataset(encoded_examples)

    formats = Counter(example["answer_format"] for example in encoded_examples)
    token_counts = Counter(example["answer_token_count"] for example in encoded_examples)
    representatives = []
    for answer_format in ("multiple_choice", "free_text"):
        match = next((example for example in encoded_examples if example["answer_format"] == answer_format), None)
        if match:
            first = match["answer_token_index"]
            representatives.append({
                "format": answer_format,
                "source": match["source"],
                "prompt_tokens_masked": first,
                "answer_tokens_supervised": match["answer_token_count"],
                "target_suffix": repr(match["target_suffix"]),
            })
    display(pd.DataFrame(representatives))
    READINESS["tokenizer"] = {"status": "PASS", "model": CFG.base_model_id, "revision": CFG.base_revision}
    READINESS["masking"] = {
        "status": "PASS",
        "examples": len(encoded_examples),
        "overlength_dropped_train_rows": len(overlength_drops),
        "formats": dict(formats),
        "answer_token_counts": dict(sorted(token_counts.items())),
    }
else:
    READINESS["tokenizer"] = {"status": "NOT_RUN"}
    READINESS["masking"] = {"status": "NOT_RUN"}

display(pd.DataFrame(READINESS).T)


## 4. Deterministic training and adapter checkpoints

Every optimizer update is logged. Loss is reported overall, by `task_type`, by `meta.source`, and by their cross-product. There is intentionally no arm breakdown.


In [ ]:
def worker_seed(worker_id):
    random.seed(CFG.seed + worker_id)
    np.random.seed(CFG.seed + worker_id)

def train_bio_knowledge(*, max_steps=None, save_adapter=True, run_label="bio_knowledge"):
    import torch
    from torch.utils.data import DataLoader
    from tqdm.auto import tqdm
    from transformers import get_scheduler

    if encoded_train is None:
        raise RuntimeError("Run the readiness/masking cell first")
    if any(gate.get("status") != "PASS" for gate in READINESS.values()):
        raise RuntimeError(f"Readiness gates did not pass: {READINESS}")
    set_all_seeds(CFG.seed)

    output_dir = ADAPTER_DIR if save_adapter else RUN_DIR / "dry_run_not_saved"
    log_dir = RUN_DIR / "logs"
    output_dir.mkdir(parents=True, exist_ok=True)
    log_dir.mkdir(parents=True, exist_ok=True)
    log_path = log_dir / f"{run_label}.jsonl"

    generator = torch.Generator().manual_seed(CFG.seed)
    loader = DataLoader(
        encoded_train,
        batch_size=CFG.micro_batch_size,
        shuffle=True,
        generator=generator,
        num_workers=CFG.num_workers,
        worker_init_fn=worker_seed if CFG.num_workers else None,
        collate_fn=lambda rows: collate_completion_batch(rows, tokenizer),
    )
    base = load_base_model(for_training=True)
    resolved_revision = base._resolved_base_revision
    model = attach_lora(base)
    trainable = [parameter for parameter in model.parameters() if parameter.requires_grad]
    optimizer = torch.optim.AdamW(trainable, lr=CFG.learning_rate, weight_decay=CFG.weight_decay)

    updates_per_epoch = math.ceil(len(loader) / CFG.gradient_accumulation_steps)
    total_updates = max_steps if max_steps is not None else updates_per_epoch * CFG.epochs
    scheduler = get_scheduler(
        CFG.scheduler,
        optimizer=optimizer,
        num_warmup_steps=round(total_updates * CFG.warmup_ratio),
        num_training_steps=total_updates,
    )
    device = next(model.parameters()).device
    optimizer.zero_grad(set_to_none=True)
    update_step = 0
    accumulation = []
    events = []
    checkpoint_dirs = []
    stop_requested = False
    epochs_to_run = CFG.epochs if max_steps is None else max(
        CFG.epochs, math.ceil(max_steps / max(1, updates_per_epoch))
    )
    progress = tqdm(total=total_updates, desc=f"Training {run_label}", unit="update")
    model.train()
    torch.cuda.reset_peak_memory_stats()

    with log_path.open("w", encoding="utf-8", newline="\n") as log_handle:
        for epoch in range(epochs_to_run):
            for batch_index, batch in enumerate(loader):
                model_inputs = {
                    key: batch[key].to(device)
                    for key in ("input_ids", "attention_mask", "labels")
                }
                loss_batch = {**batch, "labels": model_inputs["labels"]}
                with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
                    outputs = model(
                        input_ids=model_inputs["input_ids"],
                        attention_mask=model_inputs["attention_mask"],
                    )
                    loss, example_losses = answer_span_loss(outputs.logits, loss_batch)
                (loss / CFG.gradient_accumulation_steps).backward()
                accumulation.extend(zip(
                    batch["metadata"]["task_type"],
                    batch["metadata"]["source"],
                    example_losses.detach().float().cpu().tolist(),
                ))

                boundary = (
                    (batch_index + 1) % CFG.gradient_accumulation_steps == 0
                    or batch_index + 1 == len(loader)
                )
                if not boundary:
                    continue
                torch.nn.utils.clip_grad_norm_(trainable, 1.0)
                optimizer.step()
                scheduler.step()
                optimizer.zero_grad(set_to_none=True)
                update_step += 1

                by_task, by_source, by_task_source = defaultdict(list), defaultdict(list), defaultdict(list)
                for task_type, source, value in accumulation:
                    by_task[task_type].append(value)
                    by_source[source].append(value)
                    by_task_source[f"{task_type}|{source}"].append(value)
                event = {
                    "epoch": epoch,
                    "update_step": update_step,
                    "loss": float(np.mean([value for _, _, value in accumulation])),
                    "loss_by_task_type": {key: float(np.mean(values)) for key, values in sorted(by_task.items())},
                    "loss_by_source": {key: float(np.mean(values)) for key, values in sorted(by_source.items())},
                    "loss_by_task_type_source": {key: float(np.mean(values)) for key, values in sorted(by_task_source.items())},
                    "learning_rate": scheduler.get_last_lr()[0],
                }
                log_handle.write(json.dumps(event, sort_keys=True) + "\n")
                log_handle.flush()
                events.append(event)
                accumulation.clear()
                progress.update(1)
                progress.set_postfix(loss=f"{event['loss']:.4f}")

                if update_step % CFG.logging_steps == 0:
                    progress.write(str(event))
                if save_adapter and update_step % CFG.checkpoint_steps == 0 and update_step < total_updates:
                    checkpoint_dir = ADAPTER_DIR / "checkpoints" / f"step_{update_step:06d}"
                    checkpoint_dir.mkdir(parents=True, exist_ok=True)
                    model.save_pretrained(checkpoint_dir, safe_serialization=True)
                    tokenizer.save_pretrained(checkpoint_dir)
                    checkpoint_dirs.append(str(checkpoint_dir))
                if max_steps is not None and update_step >= max_steps:
                    stop_requested = True
                    break
            epoch_completed = batch_index + 1 == len(loader)
            if save_adapter and CFG.checkpoint_each_epoch and epoch_completed:
                checkpoint_dir = (
                    ADAPTER_DIR
                    / "checkpoints"
                    / f"epoch_{epoch + 1:03d}_step_{update_step:06d}"
                )
                checkpoint_dir.mkdir(parents=True, exist_ok=True)
                model.save_pretrained(checkpoint_dir, safe_serialization=True)
                tokenizer.save_pretrained(checkpoint_dir)
                checkpoint_dirs.append(str(checkpoint_dir))
            if stop_requested:
                break

    progress.close()
    if save_adapter:
        ADAPTER_DIR.mkdir(parents=True, exist_ok=True)
        model.save_pretrained(ADAPTER_DIR, safe_serialization=True)
        tokenizer.save_pretrained(ADAPTER_DIR)
    result = {
        "adapter_dir": str(ADAPTER_DIR) if save_adapter else None,
        "checkpoint_dirs": checkpoint_dirs,
        "updates": update_step,
        "log": str(log_path),
        "resolved_base_revision": resolved_revision,
        "precision_mode": CFG.precision_mode,
        "peak_gpu_bytes": torch.cuda.max_memory_allocated(),
        "events": events if max_steps is not None else None,
    }
    del model, base
    torch.cuda.empty_cache()
    return result


In [ ]:
DRY_RUN_RESULT = None
DRY_RUN_GATE = {"status": "NOT_RUN"}
if RUN_DRY_RUN:
    DRY_RUN_RESULT = train_bio_knowledge(
        max_steps=CFG.dry_run_steps,
        save_adapter=False,
        run_label="dry_run_bio_knowledge",
    )
    losses = np.asarray([event["loss"] for event in DRY_RUN_RESULT["events"]], dtype=float)
    window = max(5, len(losses) // 5)
    DRY_RUN_GATE = {
        "status": "PASS" if len(losses) == CFG.dry_run_steps and np.isfinite(losses).all() and losses[-window:].mean() < losses[:window].mean() else "FAIL",
        "steps": len(losses),
        "initial_loss": float(losses[:window].mean()),
        "final_loss": float(losses[-window:].mean()),
        "all_losses_finite": bool(np.isfinite(losses).all()),
    }
    display(DRY_RUN_GATE)
    if DRY_RUN_GATE["status"] != "PASS":
        raise RuntimeError(f"Dry-run gate failed: {DRY_RUN_GATE}")

TRAINING_RESULT = None
if RUN_TRAINING:
    if DRY_RUN_GATE["status"] != "PASS":
        raise RuntimeError("Training is blocked until the dry-run gate passes")
    TRAINING_RESULT = train_bio_knowledge(save_adapter=True)
    print("Saved bio-knowledge adapter:", TRAINING_RESULT["adapter_dir"])
else:
    print("Training disabled. Set RUN_DRY_RUN=1 and RUN_TRAINING=1 explicitly to train.")


## 5. Dev-only base-versus-adapter evaluation

MCQs use contextual answer-token logprob argmax. Exact-match tasks use deterministic generation and the existing normalized exact-match helpers. Results include Wilson 95% confidence intervals by task type and source. No test or heldout file is loaded.


In [ ]:
def load_eval_model(adapter_path=None):
    import torch
    model = load_base_model(for_training=False)
    if adapter_path is not None:
        from peft import PeftModel
        adapter_path = Path(adapter_path)
        if not (adapter_path / "adapter_config.json").is_file():
            raise FileNotFoundError(adapter_path / "adapter_config.json")
        model = PeftModel.from_pretrained(model, adapter_path, is_trainable=False)
    model.config.use_cache = True
    model.eval()
    return model

def score_dev_record(model, tokenizer, record):
    import torch
    prompt = render_plain_prompt(record)
    encoded = tokenizer(prompt, return_tensors="pt", add_special_tokens=False)
    device = next(model.parameters()).device
    encoded = {key: value.to(device) for key, value in encoded.items()}
    gold = plain_correct_target(record)

    if record["grading"] == "choice_match":
        choice_map = contextual_answer_token_ids(tokenizer, prompt, record)
        choices = plain_answer_choices(record)
        choice_ids = [choice_map[choice] for choice in choices]
        with torch.inference_mode():
            logits = model(**encoded).logits[0, -1, choice_ids].float()
        predicted = choices[int(torch.argmax(logits).item())]
        correct = predicted == gold
    elif record["grading"] == "exact_match":
        with torch.inference_mode():
            output = model.generate(
                **encoded,
                max_new_tokens=CFG.max_new_tokens,
                do_sample=False,
                pad_token_id=tokenizer.pad_token_id,
            )
        continuation = output[0, encoded["input_ids"].shape[1]:]
        predicted = tokenizer.decode(continuation, skip_special_tokens=True).strip()
        correct = exact_answers_match(predicted, gold, record)
    else:
        raise RuntimeError(f"Unsupported dev grading: {record['grading']}")

    return {
        "id": record["id"],
        "pair_id": record["pair_id"],
        "task_type": record["task_type"],
        "source": record["meta"]["source"],
        "grading": record["grading"],
        "predicted": predicted,
        "gold": gold,
        "is_correct": bool(correct),
    }

def evaluate_dev(adapter_path=None, label="base"):
    import torch
    model = load_eval_model(adapter_path)
    results = []
    for index, record in enumerate(dev_records, 1):
        results.append(score_dev_record(model, tokenizer, record))
        if index % 100 == 0:
            print(f"{label}: scored {index}/{len(dev_records)}")
    del model
    torch.cuda.empty_cache()
    return results

def accuracy_table(results, variant):
    rows = []
    groupings = [
        ("overall", lambda row: "all"),
        ("task_type", lambda row: row["task_type"]),
        ("source", lambda row: row["source"]),
    ]
    for grouping, key_fn in groupings:
        grouped = defaultdict(list)
        for result in results:
            grouped[key_fn(result)].append(result)
        for group, values in sorted(grouped.items()):
            correct = sum(value["is_correct"] for value in values)
            low, high = wilson_interval(correct, len(values))
            rows.append({
                "variant": variant,
                "grouping": grouping,
                "group": group,
                "correct": correct,
                "total": len(values),
                "accuracy": correct / len(values),
                "wilson_low": low,
                "wilson_high": high,
            })
    return rows

def adapter_candidates():
    candidates = []
    checkpoint_root = ADAPTER_DIR / "checkpoints"
    if checkpoint_root.exists():
        for pattern in ("step_*", "epoch_*"):
            candidates.extend(sorted(
                path for path in checkpoint_root.glob(pattern)
                if (path / "adapter_config.json").is_file()
            ))
    if (ADAPTER_DIR / "adapter_config.json").is_file():
        candidates.append(ADAPTER_DIR)
    return candidates


In [ ]:
DEV_RESULTS = {}
DEV_METRICS = pd.DataFrame()
SELECTED_ADAPTER_PATH = None

if RUN_DEV_EVAL:
    if tokenizer is None:
        raise RuntimeError("Run the readiness cell before evaluation")
    DEV_RESULTS["base"] = evaluate_dev(label="base")
    metric_rows = accuracy_table(DEV_RESULTS["base"], "base")

    candidates = adapter_candidates()
    explicit_adapter = os.getenv("BIO_ADAPTER_PATH", "").strip()
    if explicit_adapter:
        candidates = [Path(explicit_adapter)]
    elif not SWEEP_DEV_CHECKPOINTS and candidates:
        candidates = [ADAPTER_DIR if (ADAPTER_DIR / "adapter_config.json").is_file() else candidates[-1]]
    if not candidates:
        raise FileNotFoundError("No bio-knowledge adapter/checkpoint found")

    candidate_scores = []
    for candidate in candidates:
        label = candidate.name
        DEV_RESULTS[label] = evaluate_dev(candidate, label=label)
        rows = accuracy_table(DEV_RESULTS[label], label)
        metric_rows.extend(rows)
        overall = next(row["accuracy"] for row in rows if row["grouping"] == "overall")
        candidate_scores.append((overall, str(candidate)))
    SELECTED_ADAPTER_PATH = Path(max(candidate_scores, key=lambda value: value[0])[1])
    DEV_METRICS = pd.DataFrame(metric_rows)
    display(DEV_METRICS)
    print("Selected by dev accuracy:", SELECTED_ADAPTER_PATH)
else:
    print("Dev evaluation disabled. Set RUN_DEV_EVAL=1; test and heldout remain untouched.")


## 6. Run manifest

The manifest records the canonical split reference and hashes, per-source counts, seeds, hyperparameters, resolved base revision, adapter/checkpoints, and dev comparison when run.


In [ ]:
def counts_by_source(records):
    return dict(sorted(Counter(record["meta"]["source"] for record in records).items()))

if TRAINING_RESULT is not None:
    RUN_DIR.mkdir(parents=True, exist_ok=True)
    manifest = {
        "format_version": 1,
        "experiment": "bio_knowledge_plain_sft",
        "password_fields_used": False,
        "nonbio_included": False,
        "base": {
            "id": CFG.base_model_id,
            "requested_revision": CFG.base_revision,
            "resolved_revision": TRAINING_RESULT["resolved_base_revision"],
        },
        "seeds": {"training": CFG.seed},
        "hyperparameters": asdict(CFG),
        "lora_target_modules": list(LORA_TARGET_MODULES),
        "canonical_split": {
            "manifest_path": str(split_manifest_path),
            "manifest_sha256": sha256_file(split_manifest_path),
            "seed": split_manifest.get("seed"),
            "train_path": str(train_path),
            "train_sha256": sha256_file(train_path),
            "dev_path": str(dev_path),
            "dev_sha256": sha256_file(dev_path),
        },
        "counts": {
            "train_before_balancing": len(natural_train_records),
            "train_after_balancing": len(train_records),
            "dev": len(dev_records),
            "train_by_source_before_balancing": counts_by_source(natural_train_records),
            "train_by_source_after_balancing": counts_by_source(train_records),
            "dev_by_source": counts_by_source(dev_records),
        },
        "readiness": READINESS,
        "train_source_balancing": {
            "enabled": BALANCE_TRAIN_SOURCES,
            "max_fraction": MAX_SOURCE_FRACTION,
            "max_items_per_source": MAX_ITEMS_PER_SOURCE,
            "min_keep": MIN_KEEP_PER_SOURCE,
            "seed": DOWNSAMPLE_SEED,
            "max_largest_to_smallest_ratio": MAX_LARGEST_TO_SMALLEST_RATIO,
            "report": balance_report,
        },
        "dry_run_gate": DRY_RUN_GATE,
        "training": TRAINING_RESULT,
        "selected_adapter_by_dev": str(SELECTED_ADAPTER_PATH) if SELECTED_ADAPTER_PATH else None,
        "dev_metrics": DEV_METRICS.to_dict(orient="records") if not DEV_METRICS.empty else None,
    }
    manifest_path = ADAPTER_DIR / "training_manifest.json"
    manifest_path.write_text(json.dumps(manifest, indent=2, sort_keys=True) + "\n", encoding="utf-8")
    print("Wrote", manifest_path)
else:
    print("Manifest will be written beside adapter_bio_knowledge after training.")
